### Task 1: Load data and set up for RFM

In [8]:
import pandas as pd
df=pd.read_csv('Superstore.csv',encoding='latin1')

#conver order date to datetime (same as day 1)
df['Order Date']=pd.to_datetime(df['Order Date'],format='%d/%m/%Y')
print(df.shape)
print(df[['Customer ID','Order Date','Sales']].head())

(9800, 18)
  Customer ID Order Date     Sales
0    CG-12520 2017-11-08  261.9600
1    CG-12520 2017-11-08  731.9400
2    DV-13045 2017-06-12   14.6200
3    SO-20335 2016-10-11  957.5775
4    SO-20335 2016-10-11   22.3680


In [12]:
# Set reference date = 1 day after the last order date in the dataset
reference_date = df['Order Date'].max() + pd.Timedelta(days=1)
print("Reference Date:", reference_date)

# Group by customer, find their most recent order date
recency_df = df.groupby('Customer ID')['Order Date'].max().reset_index()
recency_df.columns = ['Customer ID', 'Last Purchase Date']

# Calculate Recency = days since last purchase
recency_df['Recency'] = (reference_date - recency_df['Last Purchase Date']).dt.days

print(recency_df.head())
print(recency_df.shape)

Reference Date: 2018-12-31 00:00:00
  Customer ID Last Purchase Date  Recency
0    AA-10315         2018-06-29      185
1    AA-10375         2018-12-11       20
2    AA-10480         2018-04-15      260
3    AA-10645         2018-11-05       56
4    AB-10015         2017-11-10      416
(793, 3)


### Task 3: Calculate Frequency

In [15]:
# Frequency = number of UNIQUE orders per customer (not just row count)
frequency_df = df.groupby('Customer ID')['Order ID'].nunique().reset_index()
frequency_df.columns = ['Customer ID', 'Frequency']

print(frequency_df.head())
print(frequency_df.shape)
print(frequency_df['Frequency'].describe())

  Customer ID  Frequency
0    AA-10315          5
1    AA-10375          9
2    AA-10480          4
3    AA-10645          6
4    AB-10015          3
(793, 2)
count    793.000000
mean       6.206810
std        2.525647
min        1.000000
25%        4.000000
50%        6.000000
75%        8.000000
max       17.000000
Name: Frequency, dtype: float64


### Task 4: Calculate Monetary

In [18]:
# Monetary = total sales per customer
monetary_df = df.groupby('Customer ID')['Sales'].sum().reset_index()
monetary_df.columns = ['Customer ID', 'Monetary']

print(monetary_df.head())
print(monetary_df.shape)
print(monetary_df['Monetary'].describe())

  Customer ID  Monetary
0    AA-10315  5563.560
1    AA-10375  1056.390
2    AA-10480  1790.512
3    AA-10645  5086.935
4    AB-10015   886.156
(793, 2)
count      793.000000
mean      2851.874884
std       2620.668723
min          4.833000
25%       1081.466000
50%       2215.002000
75%       3670.258000
max      25043.050000
Name: Monetary, dtype: float64


### Task 5: Combine Recency, Frequency, and Monetary into one RFM table

In [21]:
# Merge Recency, Frequency, and Monetary into one table
rfm_df = recency_df.merge(frequency_df, on='Customer ID').merge(monetary_df, on='Customer ID')

# Keep only relevant columns
rfm_df = rfm_df[['Customer ID', 'Recency', 'Frequency', 'Monetary']]

print(rfm_df.head())
print(rfm_df.shape)

  Customer ID  Recency  Frequency  Monetary
0    AA-10315      185          5  5563.560
1    AA-10375       20          9  1056.390
2    AA-10480      260          4  1790.512
3    AA-10645       56          6  5086.935
4    AB-10015      416          3   886.156
(793, 4)


### Task 6: Score customers using RFM segments

In [24]:
# Score Recency (lower recency = higher score, so we reverse the labels)
rfm_df['R_Score'] = pd.qcut(rfm_df['Recency'], 5, labels=[5, 4, 3, 2, 1])

# Score Frequency and Monetary (higher value = higher score)
rfm_df['F_Score'] = pd.qcut(rfm_df['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm_df['M_Score'] = pd.qcut(rfm_df['Monetary'], 5, labels=[1, 2, 3, 4, 5])

print(rfm_df.head())

  Customer ID  Recency  Frequency  Monetary R_Score F_Score M_Score
0    AA-10315      185          5  5563.560       2       2       5
1    AA-10375       20          9  1056.390       5       5       2
2    AA-10480      260          4  1790.512       1       1       3
3    AA-10645       56          6  5086.935       3       3       5
4    AB-10015      416          3   886.156       1       1       1
